# 7 — Backbone ablation inside InterGATE

This notebook evaluates message-passing backbones inside the same supervised sparse InterGATE protocol. It is an internal architecture ablation: the gate learning, Top-K pruning, sample-conditioned edge modulation, stability/fine-tuning logic and hybrid head remain those of InterGATE; only the message-passing block is changed.

Do not mix these rows with fixed-prior GNN baselines. Fixed-prior baselines are graph controls in which graph learning is disabled, whereas this notebook tests alternative backbones within the proposed sparse graph-learning framework.


## 1. Imports and project setup


In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch

from intergate.config import CFG
from intergate.utils import set_seed, cleanup_memory
from intergate.data import (
    load_expression_and_metadata,
    prepare_genes,
    encode_labels,
    cohort_split,
    scale_features,
    apply_connected_only_to_dataframe,
    make_dataloaders,
    build_regulator_features,
)
from intergate.graph_cache import get_or_build_backbone, get_or_build_Xh
from intergate.ablation import register_runtime
from intergate.backbone_ablation import (
    make_backbone_configs,
    run_backbone_ablation,
    summarize_backbone_results,
)

set_seed(CFG.SEED)
DEVICE = CFG.DEVICE
print("DEVICE:", DEVICE)


## 2. Load expression matrix, metadata and prior graph


In [ ]:
X_df, y_str, cohort = load_expression_and_metadata(
    CFG.EXPR_CSV,
    CFG.META_CSV,
    sample_col=CFG.SAMPLE_COL,
    label_col=CFG.LABEL_COL,
    cohort_col=CFG.COHORT_COL,
)
X_df_kegg, genes_kegg = prepare_genes(X_df)
y, classes, label_map = encode_labels(y_str)
n_classes = len(classes)
print("Classes:", classes)


In [ ]:
edge_index, edge_weight, edge_type = get_or_build_backbone(
    genes_kegg,
    cache_dir=CFG.PIPELINE_CACHE_DIR,
    force_rebuild=CFG.FORCE_REBUILD_HURI_CACHE,
    use_omnipath=CFG.USE_OMNIPATH,
    use_huri=CFG.USE_HURI,
)
print(f"Backbone: {edge_index.shape[1]} directed edges, {len(genes_kegg)} genes before connected-only filtering")


## 3. Connected-only filtering, cohort-aware split and scaling


In [ ]:
if CFG.CONNECTED_ONLY:
    X_df_kegg, edge_index, edge_weight, edge_type, genes_kegg = apply_connected_only_to_dataframe(
        X_df_kegg,
        edge_index,
        edge_weight,
        edge_type,
        genes_kegg,
    )

train_idx, val_idx, test_idx = cohort_split(
    cohort,
    y,
    train_cohort_frac=0.80,
    val_size=CFG.VAL_SIZE,
    seed=CFG.SEED,
)

Xs_gene = scale_features(
    X_df_kegg,
    train_idx,
    mode=CFG.SCALE_MODE,
    use_quantile=CFG.USE_QUANTILE,
)
print(f"Xs_gene: {Xs_gene.shape} dtype={Xs_gene.dtype}")


## 4. Regulator-summary covariates and tensors


In [ ]:
try:
    Xs_graph, graph_feat_names = get_or_build_Xh(
        Xs_gene,
        genes_kegg,
        edge_index,
        cache_dir=CFG.PIPELINE_CACHE_DIR,
        force_rebuild=False,
        stats=CFG.REG_STATS,
        min_targets=CFG.REG_MIN_GENES,
        max_regulators=CFG.REG_MAX_REGULATORS,
    )
except Exception as exc:
    print("[WARN] Could not load X_h from cache; recomputing regulator features:", exc)
    Xs_graph, graph_feat_names = build_regulator_features(
        Xs_gene,
        genes_kegg,
        edge_index,
        add_features=CFG.ADD_REGULATOR_FEATURES,
        stats=CFG.REG_STATS,
        min_targets=CFG.REG_MIN_GENES,
        max_regulators=CFG.REG_MAX_REGULATORS,
    )

print(f"Xs_graph: {Xs_graph.shape} | features: {len(graph_feat_names)}")


In [ ]:
dl_tr, dl_va, dl_te = make_dataloaders(
    Xs_gene,
    Xs_graph,
    y,
    train_idx,
    val_idx,
    test_idx,
    batch_size=CFG.BATCH_SIZE,
)

edge_index_t = torch.as_tensor(edge_index, dtype=torch.long, device=DEVICE)
edge_weight_t = torch.as_tensor(edge_weight, dtype=torch.float32, device=DEVICE)
edge_type_t = torch.as_tensor(edge_type, dtype=torch.long, device=DEVICE)

print("edge_index_t:", tuple(edge_index_t.shape), "edge_weight_t:", tuple(edge_weight_t.shape))


## 5. Register runtime objects for the ablation engine


In [ ]:
register_runtime(
    Xs_gene=Xs_gene,
    genes_kegg=genes_kegg,
    X_h=Xs_graph,
    y=y,
    n_classes=n_classes,
    label_map=label_map,
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx,
    edge_index_t=edge_index_t,
    edge_weight_t=edge_weight_t,
    edge_type_t=edge_type_t,
)


## 6. Run the backbone-replacement ablation

The default below is a fast development screen. For a paper-level run, increase the number of seeds and set `do_stability=True` once the selected backbone set is fixed.


In [ ]:
BACKBONE_SEEDS = [1, 42, 1234]
BACKBONES = ("gat", "sage", "gin", "graph_transformer")

configs = make_backbone_configs(
    backbones=BACKBONES,
    do_stability=False,
    do_pretrain=False,
)

df_backbones = run_backbone_ablation(
    configs=configs,
    seeds=BACKBONE_SEEDS,
    save_root=str(PROJECT_ROOT / "artifacts_backbone_ablation"),
    force_save=False,
)

summary_backbones = summarize_backbone_results(df_backbones)
display(summary_backbones)


## 7. Save a compact manuscript-ready summary


In [ ]:
out_dir = PROJECT_ROOT / "artifacts_backbone_ablation"
out_dir.mkdir(parents=True, exist_ok=True)
summary_path = out_dir / "backbone_ablation_summary_for_manuscript.csv"
summary_backbones.to_csv(summary_path, index=False)
print("Saved:", summary_path)
